# Bronze layer walkthrough

This notebook mirrors `scripts/bronze.py`. It stores the complete CoinGecko and Frankfurter responses in two separate raw JSONB tables. No coin prices or exchange rates are normalized here.

Run from the repository root after configuring the `DB_*` variables in `.env`.

## 1. Imports and database connection

In [ ]:
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.postgresql import JSONB
from sqlalchemy.engine import URL

repo_root = Path.cwd()
if not (repo_root / '.env').exists():
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

def required_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'Missing required environment variable: {name}')
    return value

engine = create_engine(
    URL.create(
        drivername='postgresql+psycopg2',
        username=required_env('DB_USER'),
        password=required_env('DB_PASSWORD'),
        host=required_env('DB_HOST'),
        port=int(os.getenv('DB_PORT', '5432')),
        database=required_env('DB_NAME'),
    ),
    connect_args={'sslmode': 'require'},
)

## 2. Fetch the complete CoinGecko response

In [ ]:
coin_response = requests.get(
    'https://api.coingecko.com/api/v3/simple/price',
    params={
        'ids': 'bitcoin,ethereum,ripple,solana,doge',
        'vs_currencies': 'usd',
        'include_last_updated_at': 'true',
    },
    timeout=30,
)
coin_response.raise_for_status()
coin_payload = coin_response.json()
coin_payload

## 3. Store the untouched CoinGecko payload

In [ ]:
batch_id = str(uuid.uuid4())
extracted_at = pd.Timestamp(datetime.now(timezone.utc))

coin_raw = pd.DataFrame([{
    'batch_id': batch_id,
    'extracted_at': extracted_at,
    'raw_payload': coin_payload,
}])

coin_raw[['batch_id', 'extracted_at']]

In [ ]:
coin_raw.to_sql(
    name='bronze_coin_gecko',
    con=engine,
    if_exists='append',
    index=False,
    dtype={'raw_payload': JSONB},
)
print(f'Inserted CoinGecko bronze batch {batch_id}.')

## 4. Fetch the complete currency-rate response

In [ ]:
currency_response = requests.get(
    'https://api.frankfurter.dev/v1/latest',
    params={'base': 'USD', 'symbols': 'IDR'},
    timeout=30,
)
currency_response.raise_for_status()
currency_payload = currency_response.json()
currency_payload

## 5. Store the untouched currency payload

In [ ]:
currency_raw = pd.DataFrame([{
    'batch_id': batch_id,
    'extracted_at': extracted_at,
    'raw_payload': currency_payload,
}])

currency_raw.to_sql(
    name='bronze_currency_rate',
    con=engine,
    if_exists='append',
    index=False,
    dtype={'raw_payload': JSONB},
)
print(f'Inserted currency bronze batch {batch_id}.')

## 6. Read both raw tables back

In [ ]:
stored_coin = pd.read_sql(
    text('SELECT batch_id, extracted_at, raw_payload FROM bronze_coin_gecko WHERE batch_id = :batch_id'),
    engine,
    params={'batch_id': batch_id},
)
stored_currency = pd.read_sql(
    text('SELECT batch_id, extracted_at, raw_payload FROM bronze_currency_rate WHERE batch_id = :batch_id'),
    engine,
    params={'batch_id': batch_id},
)

print('CoinGecko rows:', len(stored_coin))
print('Currency rows:', len(stored_currency))
print('CoinGecko payload keys:', sorted(stored_coin.loc[0, 'raw_payload'].keys()))
print('Currency payload keys:', sorted(stored_currency.loc[0, 'raw_payload'].keys()))

The two bronze tables contain raw JSONB plus only the shared batch metadata. Normalization happens in silver.